In [ ]:
import re
import json
from bs4 import BeautifulSoup
from typing import Optional

# Regex patterns for data extraction
EURO_RE = re.compile(r"€\s*([\d,]+)")
AREA_RE = re.compile(
    r"([\d,.]+)\s*(?:m²|m2|sqm|sq\.?\s*m)\b", 
    re.IGNORECASE
)

def extract_area_m2(soup: BeautifulSoup) -> Optional[float]:
    page_text = soup.get_text("\n", strip=True)
    page_text = (
        page_text
        .replace("²", "²")
        .replace("m2", "m²")
        .replace("M2", "m²")
    )
    m = AREA_RE.search(page_text)
    if not m:
        return None
    return float(m.group(1).replace(",", ""))

def extract_views(soup: BeautifulSoup) -> Optional[int]:
    page_text = soup.get_text("\n", strip=True)
    
    # Try specific selectors first
    views_selectors = [
        '[data-testid="views-count"]',
        '[data-testid="view-count"]', 
        '.views-count',
        '.view-count',
        '[class*="view"][class*="count"]',
    ]
    
    for selector in views_selectors:
        try:
            elem = soup.select_one(selector)
            if elem:
                views_text = elem.get_text(strip=True)
                views_num = re.search(r'(\d{1,3}(?:,\d{3})*|\d+)', views_text)
                if views_num:
                    return int(views_num.group(1).replace(',', ''))
        except:
            pass
    
    # Fallback to text search
    m = re.search(r'(\d{1,3}(?:,\d{3})*|\d+)\s*(?:view|viewing)s?\b', page_text, re.IGNORECASE)
    if m:
        return int(m.group(1).replace(',', ''))
    
    return None

def extract_fields(html: str) -> dict:
    soup = BeautifulSoup(html, "lxml")
    
    def txt(sel):
        el = soup.select_one(sel)
        return el.get_text(" ", strip=True) if el else None
    
    # Title
    h1 = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else None
    
    # Address (multiple fallback selectors)
    address = (
        txt('[data-testid="address"]') or
        txt('[data-testid="header-subtitle"]') or
        None
    )
    if not address and title and "," in title:
        parts = [p.strip() for p in title.split(",") if p.strip()]
        if len(parts) >= 2:
            address = ", ".join(parts[-2:])
    
    # Price
    price_text = txt('[data-testid="price"]') or txt('[data-testid="header-price"]')
    price_eur = None
    if price_text:
        m = EURO_RE.search(price_text)
        if m:
            price_eur = int(m.group(1).replace(",", ""))
    
    # Beds/Baths from text
    page_text = soup.get_text("\n", strip=True)
    beds = baths = None
    m = re.search(r"\b(\d+)\s*Bed\b", page_text, re.IGNORECASE)
    if m:
        beds = int(m.group(1))
    m = re.search(r"\b(\d+)\s*Bath\b", page_text, re.IGNORECASE)
    if m:
        baths = int(m.group(1))
    
    # Area
    area_m2 = extract_area_m2(soup)
    
    # Price per m2
    price_per_m2_eur = None
    m = re.search(r"Price\s*per\s*m²\s*[:\-]?\s*€\s*([\d,]+)", page_text, re.IGNORECASE)
    if m:
        price_per_m2_eur = int(m.group(1).replace(",", ""))
    elif price_eur is not None and area_m2:
        price_per_m2_eur = round(price_eur / area_m2)
    
    # BER rating
    ber_rating = None
    ber_el = soup.select_one('[aria-label*="BER"], [data-testid*="ber"] [aria-label]')
    if ber_el and ber_el.get("aria-label"):
        m = re.search(r"\b([A-G]\d?)\b", ber_el["aria-label"].upper())
        if m:
            ber_rating = m.group(1)
    
    # VIEWS - CRITICAL FOR AD PERFORMANCE
    views = extract_views(soup)
    
    return {
        "title": title,
        "address": address,
        "ber_rating": ber_rating,
        "beds": beds,
        "baths": baths,
        "area_m2": area_m2,
        "price_eur": price_eur,
        "price_per_m2_eur": price_per_m2_eur,
        "views": views  # ← This gets "4,462" → 4462 numeric
    }

print("✅ extract_fields function loaded - ready to scrape views data")

In [ ]:
# STEP 1A: GET FRESH DAF T.IE NEW HOMES LISTINGS (SELENIUM)
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
from urllib.parse import urljoin
import time

print("Loading daft.ie with Selenium for fresh listings...")

# Quick Selenium setup for listing pages
options = Options()
options.add_argument("--headless")  # No browser window
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

fresh_urls = []

try:
    # Load new homes page
    driver.get("https://www.daft.ie/new-homes-for-sale/ireland/")
    time.sleep(5)  # Wait for dynamic load
    
    # Multiple selector strategies for daft.ie listings
    selectors = [
        'a[href*="/for-sale/"]',
        '.PropertyCard_Link',
        '[data-testid="listing-link"]', 
        'h2 a[href*="/for-sale/"]',
        '.SearchResultsCard_Link a',
        'a[class*="card"]',
        '.card-title a',
        '.listing-title a'
    ]
    
    all_links = []
    for selector in selectors:
        links = driver.find_elements(By.CSS_SELECTOR, selector)
        all_links.extend([link.get_attribute('href') for link in links if link.get_attribute('href')])
        time.sleep(1)
    
    # Filter valid property URLs
    for href in all_links[:100]:  # Check first 100 links
        if href and ('/for-sale/' in href or '/property/' in href):
            parsed = href.split('/')[-2:]  # Extract listing ID
            if len(parsed) == 2 and parsed[0].isdigit():
                fresh_urls.append(href)
    
    # Remove duplicates and limit
    fresh_urls = list(set(fresh_urls))[:30]  # 30 fresh URLs max
    
    print(f"✅ Found {len(fresh_urls)} fresh property URLs")
    
finally:
    driver.quit()

# Save fresh URLs
if fresh_urls:
    df_urls = pd.DataFrame({'url': fresh_urls})
    df_urls.to_csv('fresh_urls_today.csv', index=False)
    print("\nSample fresh URLs:")
    for i, url in enumerate(df_urls['url'].head(), 1):
        print(f"{i:2d}. {url}")
else:
    print("❌ No URLs found - using your original CSV as fallback")
    df_urls = pd.read_csv(r"C:\Users\danie\OneDrive\Python\Ciaran\Task 2\xxx\new_homes\daft_listings_newhomes_20260404_145356.csv")[['url']].dropna()

print(f"\nReady for scraper - {len(df_urls)} URLs prepared")

In [ ]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Load your 49 PROVEN URLs
df_urls = pd.read_csv(r"C:\Users\danie\OneDrive\Python\Ciaran\Task 2\xxx\new_homes\daft_listings_newhomes_20260404_145356.csv")
development_urls = df_urls["url"].dropna().tolist()[:5]  # Test first 5
print(f"Testing {len(development_urls)} URLs")

# Selenium setup
options = Options()
options.add_argument("--window-size=1400,900")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

rows = []

for i, url in enumerate(development_urls):
    print(f"\n[{i+1}/5] {url}")
    driver.get(url)
    time.sleep(6)
    
    html = driver.page_source
    
    # Always create basic data structure
    data = {
        "title": "Not found",
        "address": "Not found", 
        "views": None,
        "price_eur": None,
        "beds": None,
        "baths": None
    }
    
    try:
        extracted = extract_fields(html)
        # Only use fields that actually exist
        for key, value in extracted.items():
            if value is not None:
                data[key] = value
    except:
        print("  extract_fields failed - using defaults")
    
    data["url"] = url
    data["scrape_time"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    rows.append(data)
    
    print(f"  Title: {data.get('title', 'None')[:50]}...")
    print(f"  Views: {data.get('views', 'None')}")

driver.quit()

# Create DataFrame safely
df = pd.DataFrame(rows)
print(f"\n✅ Created df with {len(df)} rows")
print("Columns:", df.columns.tolist())
print("\nFirst row:")
print(df.iloc[0])

In [ ]:
import re
import pandas as pd

def standardise_county(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\s+(City Centre|City Center|City|Town)\s*$", "", s, flags=re.IGNORECASE).strip()

    s_up = s.upper().replace(" ", "")

    if re.match(r"^(DUBLIN\d*|DUB\d+)$", s_up):
        return "Dublin"

    s = re.sub(r"^(Co\.?|County)\s+", "", s, flags=re.IGNORECASE).strip()

    mapping = {
        "CORK": "Cork", "KILDARE": "Kildare", "WEXFORD": "Wexford", "WICKLOW": "Wicklow",
        "WESTMEATH": "Westmeath", "MEATH": "Meath", "GALWAY": "Galway", "LOUTH": "Louth",
        "WATERFORD": "Waterford", "LIMERICK": "Limerick", "KILKENNY": "Kilkenny",
        "LAOIS": "Laois", "OFFALY": "Offaly", "SLIGO": "Sligo", "CLARE": "Clare",
        "TIPPERARY": "Tipperary", "KERRY": "Kerry", "CAVAN": "Cavan", "MAYO": "Mayo",
        "LEITRIM": "Leitrim", "ANTRIM": "Antrim", "DONEGAL": "Donegal", "MONAGHAN": "Monaghan",
        "ROSCOMMON": "Roscommon", "CARLOW": "Carlow",
    }

    return mapping.get(s.upper(), s)

print("Columns after scraping:", df.columns.tolist())
print(f"Scraped {len(df)} fresh properties")

# Safe address extraction
if 'address' in df.columns and df['address'].notna().any():
    df["county"] = (
        df["address"]
        .fillna("")
        .astype(str)
        .str.split(",")
        .str[-1]
        .str.strip()
    )
    df["county_standard"] = df["county"].apply(standardise_county)
    print("Counties standardized successfully")
else:
    print("No address data yet - skipping county extraction")
    df["county"] = "Unknown"
    df["county_standard"] = "Unknown"

print("\nSample counties:")
print(df[["title", "address"]])

In [ ]:
# Province mapping for Irish counties
province_map = {
    "Dublin": "Leinster",
    "Kildare": "Leinster", 
    "Wicklow": "Leinster",
    "Wexford": "Leinster",
    "Kilkenny": "Leinster",
    "Laois": "Leinster",
    "Offaly": "Leinster",
    "Meath": "Leinster",
    "Westmeath": "Leinster",
    "Carlow": "Leinster",
    "Louth": "Leinster",
    "Cork": "Munster",
    "Waterford": "Munster",
    "Limerick": "Munster",
    "Kerry": "Munster",
    "Tipperary": "Munster",
    "Clare": "Munster",
    "Galway": "Connacht",
    "Mayo": "Connacht",
    "Sligo": "Connacht",
    "Roscommon": "Connacht",
    "Leitrim": "Connacht",
    "Donegal": "Ulster",
    "Monaghan": "Ulster",
    "Cavan": "Ulster",
    "Antrim": "Ulster",
}

# Map counties to provinces (safe handling)
df['province'] = df['county_standard'].map(province_map).fillna('Unknown')

print("Province mapping complete")
print(f"Properties by province:")
print(df['province'].value_counts().head(10))

print("\nSample data with provinces:")
print(df[['title', 'address', 'county_standard', 'province', 'views']].head())

In [ ]:
# VIEWS PERFORMANCE ANALYSIS - FRESH PROPERTIES
print("VIEWS PERFORMANCE ANALYSIS")
print("=" * 60)
print(f"Analyzing {len(df)} fresh scraped properties")

print("Current columns:", df.columns.tolist())

# Clean views column safely
df['views_clean'] = pd.to_numeric(df['views'], errors='coerce')

# Initialize performance columns
df['views_rank'] = float('nan')
df['views_percentile'] = float('nan')
df['views_category'] = 'No Data'

# Full analysis if views data exists
if df['views_clean'].notna().sum() > 0:
    print(f"\n✅ Found views data for {df['views_clean'].notna().sum()} properties")
    
    # Basic statistics
    views_stats = df['views_clean'].describe()
    print("\nViews Statistics:")
    print(views_stats.round(0))
    
    # Performance rankings
    df['views_rank'] = df['views_clean'].rank(ascending=False)
    df['views_percentile'] = (df['views_clean'].rank(pct=True) * 100).round(1)
    
    # Performance categories
    df['views_category'] = pd.cut(df['views_clean'].fillna(0), 
                                 bins=[0, 100, 500, 1000, 5000, float('inf')],
                                 labels=['<100', '100-500', '500-1K', '1K-5K', '5K+'])
    
    # Key metrics
    print(f"\nKEY PERFORMANCE METRICS:")
    print(f"Average views: {df['views_clean'].mean():,.0f}")
    print(f"Median views: {df['views_clean'].median():,.0f}")
    print(f"Top performer: {df['views_clean'].max():,.0f} views")
    print(f"Top 10% threshold: {df['views_clean'].quantile(0.9):,.0f} views")
    
    # Views by province
    print(f"\nTOP PROVINCES BY VIEWS:")
    views_province = df.groupby('province')['views_clean'].agg(['mean', 'count']).round(0)
    views_province['mean'] = views_province['mean'].fillna(0)
    print(views_province.sort_values('mean', ascending=False).head(10))
    
    # Top 5 performers
    print(f"\n🏆 TOP 5 FRESH PROPERTIES BY VIEWS:")
    top5 = df.nlargest(5, 'views_clean')[['title', 'views_clean', 'views_rank', 'province']]
    print(top5)
    
else:
    print("No views data found yet - scraper may need to run first")

print("\n✅ VIEWS ANALYSIS COLUMNS ADDED:")
print("- views_clean (numeric views)")
print("- views_rank (1 = best performer)")
print("- views_percentile (% ranking)")
print("- views_category (performance tiers)")

print("\nReady for export with complete ad performance analysis!")

In [ ]:
from pathlib import Path
from datetime import datetime

# Output directory
OUTPUT_DIR = Path(r"C:\Users\danie\OneDrive\Python\Ciaran\Task 2\xxx\new_homes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# MAIN FILE - Complete analysis with fresh views data
main_file = OUTPUT_DIR / f"daft_fresh_views_analysis_{ts}.csv"
df.to_csv(main_file, index=False)

# VIEWS PERFORMANCE SUMMARY
views_summary = pd.DataFrame({
    'Metric': ['Total Fresh Properties', 'Properties with Views', 'Average Views', 
               'Median Views', 'Top Performer Views', 'Top 10% Threshold'],
    'Value': [
        len(df),
        df['views_clean'].notna().sum(),
        df['views_clean'].mean(),
        df['views_clean'].median(),
        df['views_clean'].max(),
        df['views_clean'].quantile(0.9)
    ]
})
views_summary.to_csv(OUTPUT_DIR / f"views_performance_summary_{ts}.csv", index=False)

# TOP 20 PERFORMERS
top_20 = df.nlargest(20, 'views_clean')[
    ['title', 'address', 'views_clean', 'views_rank', 'views_category', 
     'province', 'price_eur', 'county_standard']
].copy()
top_20.to_csv(OUTPUT_DIR / f"top_20_ad_performers_{ts}.csv", index=False)

# VIEWS BY PROVINCE
views_province = (df.groupby(['province', 'county_standard'])['views_clean']
                 .agg(['count', 'mean', 'median'])
                 .round(0)
                 .reset_index()
                 .sort_values('mean', ascending=False))
views_province.to_csv(OUTPUT_DIR / f"views_by_province_{ts}.csv", index=False)

print(f"✅ COMPLETE ANALYSIS EXPORTED:")
print(f"Main file: {main_file}")
print(f"Views summary: views_performance_summary_{ts}.csv")
print(f"Top 20: top_20_ad_performers_{ts}.csv") 
print(f"Province analysis: views_by_province_{ts}.csv")

print(f"\nMain CSV contains {len(df)} fresh properties with:")
print("- views_clean: numeric view counts")
print("- views_rank: 1 = best performing ")

In [ ]:
print("DAFT.IE AD PERFORMANCE DASHBOARD")
print("=" * 60)
print(f"Analysis Date: April 07, 2026")
print(f"Fresh Properties Analyzed: {len(df)}")
print()

# PERFORMANCE OVERVIEW
print("📊 PERFORMANCE OVERVIEW")
if 'views_clean' in df.columns:
    print(f"Total Views Captured: {df['views_clean'].sum():,.0f}")
    print(f"Average Views per Ad: {df['views_clean'].mean():,.0f}")
    print(f"Best Performing Ad: {df['views_clean'].max():,.0f} views")
    print(f"Properties in Top 10%: {len(df[df['views_clean'] >= df['views_clean'].quantile(0.9)])}")
else:
    print("No views data available")

print()

# TOP REGIONS
print("🏠 TOP REGIONS BY VIEWS")
if 'province' in df.columns and 'views_clean' in df.columns:
    region_perf = df.groupby('province')['views_clean'].agg(['mean', 'count']).round(0)
    print(region_perf.sort_values('mean', ascending=False).head())
else:
    print("Province data not available")

print()

# AD PERFORMANCE TIERS
print("📈 AD PERFORMANCE TIERS")
if 'views_category' in df.columns:
    tier_counts = df['views_category'].value_counts()
    for category, count in tier_counts.items():
        print(f"{category}: {count}")
else:
    print("No category data available")